# OCR(Optical Character Recognition)
    -  이미지 속의 문자들을 컴퓨터가 인식할 수 있도록 디지털 문자로 변환해주는 기술입니다.
    문서를 디지털화 하는 가장 기본적인 기술로 자리잡았으며, 이미지, PDF, 문서속 문자를 인식하고
    편집 및 검색이 가능하게 하는 AI기반의 기술을 의미합니다.

In [1]:
from dotenv import load_dotenv
import os

from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-qe-eR7H_eASs


## 1. llm 준비

In [2]:
ocr_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [3]:
# base64: 컴퓨터에서 8비트 바이너리(이진) 데이터인 이미지, 실행파일 등을 인쇄 가능한
# 64개의 아스키코드,(A-Z, a-z, 0-9, +, /)로만 변환하여 안전하게 전송 또는 저장하는 인코딩

import base64

def encode_image(image_path):
    with open(image_path, "rb") as image:
        return base64.b64encode(image.read()).decode("utf-8")

In [4]:
base_path = r"D:\back_0900_kdh\python\resource"
receipt_name = r"receipt_01.png"
receipt_path = os.path.join(base_path, receipt_name)

receipt_path

'D:\\back_0900_kdh\\python\\resource\\receipt_01.png'

In [5]:
base64_receipt = encode_image(receipt_path)

In [6]:
base64_receipt

'...'

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an advanced vision model.

        Your goal is to extract ALL text from the image.

        IMPORTANT:
        - Do NOT miss any text
        - If text is unclear, infer the most likely correct text
        - Preserve all lines, even unusual ones
        - Do NOT drop promotional or "증정품" lines
        """),
            ("human", [
                {
                    "type": "text",
                    "text": """
        Extract ALL visible text from this receipt.

        Rules:
        - Include EVERY line (even if it looks unimportant)
        - Keep line breaks
        - Do NOT omit anything
        - If a word is unclear, guess the most likely correct word

        Output ONLY the extracted text.
        """
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "data:image/jpeg;base64,{encoded_image}"
                    }
                }
            ])
        ])

In [8]:
ocr_chain = prompt | ocr_llm

In [9]:
result = ocr_chain.invoke({
    "encoded_image": base64_receipt
})

In [10]:
print(result.content)

2026/03/01 정*비 NO:14002
*정부방침에 의해 교환/환불은
반드시 영수증을 지참하셔야 하며,
카드결제는 30일(03월31일)이내
카드와 영수증 지참 시 가능합니다
선도민감상품 및 일부상품 제외
----------------------------------------
에그소시지샌드 1 2,800
참치김치김밥 1 3,500
정성가득비빔밥 1 4,600
합계수량/금액 3 10,900
KT 멤버십 할인 -1,000
----------------------------------------
과세 매출 9,000
부가세 900
합 계 9,900
GS QR/바코드 결제할인 -3,000
모바일상품권 3,400
현 금 3,500


## JSON 형태로 변환

In [11]:
from typing import List
from pydantic import BaseModel, Field

In [12]:
class ReceiptItem(BaseModel):
    name: str = Field(description="The exact product name from the text. Copy verbatim.")
    count: int = Field(description="The quantity.")
    price: int = Field(description="The price text exactily as printed (e.g., '1,600' or '증정품').")

class ReceiptInfo(BaseModel):
    store_name: str = Field(description="Store name")
    date: str = Field(description="YYYY-MM-DD")
    items: List[ReceiptItem] = Field(description="List of purchansed items.")
    amount: int = Field(description="Discounted price")
    total_amount: int = Field(description="Price after discount")

In [13]:
parse_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 리턴값은 무조건 ReceiptInfo 타입
structured_llm = parse_llm.with_structured_output(ReceiptInfo)

## ※문제점

In [14]:
json_ocr_chain = prompt | structured_llm

In [15]:
json_ocr_result = json_ocr_chain.invoke({
    "encoded_image": base64_receipt
})

D:\back_0900_kdh\python\workspace\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReceiptInfo(store_name='...9900, total_amount=9900), input_type=ReceiptInfo])
  return self.__pydantic_serializer__.to_python(


In [16]:
json_ocr_result

ReceiptInfo(store_name='정*비', date='2026-03-01', items=[ReceiptItem(name='에그소시지샌드', count=1, price=2800), ReceiptItem(name='참치김치김밥', count=1, price=3500), ReceiptItem(name='정성가득비빔밥', count=1, price=4600), ReceiptItem(name='합계수량/금액', count=3, price=10900), ReceiptItem(name='KT 멤버십 할인', count=1, price=-1000), ReceiptItem(name='과세 매출', count=1, price=9000), ReceiptItem(name='부가세', count=1, price=900), ReceiptItem(name='합 계', count=1, price=9900), ReceiptItem(name='GS QR/바코드 결제할인', count=1, price=-3000), ReceiptItem(name='모바일상품권', count=1, price=3400), ReceiptItem(name='현 금', count=1, price=3500)], amount=9900, total_amount=9900)

### 문제 해결

### 1. 프롬프트 생성

In [17]:
parsing_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a deterministic parser.

        STRICT RULES:
        - Each line represents EXACTLY one item.
        - NEVER merge lines.
        - NEVER split lines.
        - NEVER infer or guess missing values.
        - Copy text exactly as written.

        You must process input line-by-line.
        """
            ),
            (
                "human",
                """
        Here is the raw receipt text:

        {receipt_text}

        Instructions:
        1. Only extract lines that match this pattern:
           [product name] [count] [price or '증정품']

        2. Ignore all other lines.

        3. Output must be JSON array:
        [
          {{"name": "...", "count": ..., "price": "..."}}
        ]

        4. Do not change numbers.
        5. Do not duplicate items.
        6. Do not create new items.

        Output JSON only.
        """
    )
])

### 2. 체인 생성

In [18]:
def transform_output(input_messages):
    return {"receipt_text": input_messages.content}

def transform_output_model_dump(input_messages):
    return input_messages.model_dump()

In [19]:
parse_chain = parsing_prompt | structured_llm

In [20]:
final_chain = ocr_chain | RunnableLambda(transform_output) | parse_chain | RunnableLambda(transform_output_model_dump)

In [21]:
final_result = final_chain.invoke({
    "encoded_image": base64_receipt
})

D:\back_0900_kdh\python\workspace\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReceiptInfo(store_name='...1000, total_amount=9900), input_type=ReceiptInfo])
  return self.__pydantic_serializer__.to_python(


In [22]:
final_result

{'store_name': '정*비',
 'date': '2026-03-01',
 'items': [{'name': '에그소시지샌드', 'count': 1, 'price': 2800},
  {'name': '참치김치김밥', 'count': 1, 'price': 3500},
  {'name': '정성가득비빔밥', 'count': 1, 'price': 4600}],
 'amount': 1000,
 'total_amount': 9900}

## 객체 탐지

In [23]:
base_path = r"C:\back_0900_kdh\python\resource"

penguin_name = r"penguin.jpg"
cat_name = r"cat.png"

penguin_path = os.path.join(base_path, penguin_name)
cat_path = os.path.join(base_path, cat_name)

base64_penguin = encode_image(penguin_path)
base64_cat = encode_image(cat_path)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\back_0900_kdh\\python\\resource\\penguin.jpg'

In [25]:
classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an object detector.

        You must output exactly:
        - List all visible objects in the image.
        - ["cat"]

        Rules:
        - Output must as be JSON array
        - Use simple nouns
        - No duplicates
    """),
    ("human", [
        {
            "type": "text",
            "text": """
                List all objects in the image.

                Return format example:
                ["cat", "dog"]

                Do not add anything else.
            """
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,{encoded_image}"
            }
        }
    ])
])

In [26]:
ocr_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [31]:
import json

def json_parser(input_messages):
    return json.loads(input_messages.content)

In [32]:
classifier_chain classifier_prompt | ocr_llm | RunnableLambda(json_parser)

SyntaxError: invalid syntax (4193252586.py, line 1)

In [28]:
cat_result = classifier_chain.invoke({
    "encoded_image": base64_cat
})

NameError: name 'classifier_chain' is not defined

In [29]:
cat_result.content

NameError: name 'cat_result' is not defined

In [30]:
import json

json.loads(cat_result.content)

NameError: name 'cat_result' is not defined

In [ ]:
brief_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an expert visual scene analyst specification.

        Your task:
        - Analyze the provided image deeply and describe the scene in detail.

        Rules:
        - **Focus on:** Objects, actions, relationships, and precise colors (especially hair color, clothing color, accessories, and background colors).
        - **Detailed Breakdown:** Do not generalize. Identify specific entities and break down their appearance, posture, and environmental details.
        - **Length Rule:** Please provide a detailed explanation. The response must be at least 10 lines long. Ensure each logical point is separated by a line break to maintain readability.
        - **Constraint:** Do NOT guess unknown facts or speculate. Describe only what is visible.
        - **Language:** You must answer in Korean.
    """),
    ("human", [
        {
            "type": "text",
            "text": "Describe what is happening in this image in full detail, focusing on appearance, actions, colors, and relationships."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,{encoded_image}"
            }
        }
    ])
])

In [ ]:
brief_chain = brief_prompt | brief_llm

In [ ]:
yujung_name = "brief_yujung.png"
yujung_path = os.path.join(base_path, yujung_name)
base64_yujung = encode_image (yujung_path)

In [ ]:
brief_result = brief_chain.invoke({
    "encoded_image": base64_yujung
})

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a question-answering assistant.

        Answer based ONLY on the given description.
        Do NOT assume anything beyond the description.
    """),
    ("human", """
        Description:
        {description}

        Question:
        {question}
    """)
    ])

In [33]:
qa_chain = qa_prompt | brief_llm

NameError: name 'qa_prompt' is not defined

In [34]:
qa_result = qa_chain.invoke({
    "description": brief_result.content,
    "question": "남자가 입은 후드티의 색깔은?"
})

NameError: name 'qa_chain' is not defined

In [ ]:
def extract_question(input_value):
    return input_value["question"]

def format_qa_input(input_value):
    return {
        "description": input_value["description"].content,
        "question": input_value["question"]
    }

In [ ]:
final_qa_chain = (
    {
        "description": brief_chain,
        "question": RunnableLambda(extract_question)
    }
    | RunnableLambda(format_qa_input)
    | qa_chain
)

In [ ]:
final_qa_result = final_qa_chain.invoke({
    "encoded_image": base64_yujung,
    "question": "남자가 입고 있는 후드티의 색깔은?"
})

In [ ]:
final_qa_result.content